### Two-layer NN

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# Define the neural network class
class TwoLayerNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(TwoLayerNet, self).__init__()
        # Define the layers
        self.fc1 = nn.Linear(input_size, hidden_size)  # First layer (input to hidden)
        self.fc2 = nn.Linear(hidden_size, output_size)  # Second layer (hidden to output)
    
    def forward(self, x):
        # Forward pass: input -> hidden -> output
        x = F.relu(self.fc1(x))  # Apply ReLU activation after the first layer
        x = self.fc2(x)  # Output layer (no activation for raw logits)
        return x

# Training function
def train(model, criterion, optimizer, X_train, y_train, epochs=1000):
    for epoch in range(epochs):
        # Forward pass: Compute predicted outputs
        outputs = model(X_train)
        
        # Compute loss
        loss = criterion(outputs, y_train)
        
        # Backward pass: compute gradient of loss w.r.t. model parameters
        optimizer.zero_grad()  # Clear the gradients
        loss.backward()        # Compute gradients
        
        # Update model parameters
        optimizer.step()

        # Print loss every 100 epochs
        if (epoch + 1) % 100 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')
            
# Example usage:

# Dummy data
X_train = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]], dtype=torch.float32)  # XOR inputs
y_train = torch.tensor([0, 1, 1, 0], dtype=torch.long)  # XOR outputs (labels)

# Define the network, loss function, and optimizer
input_size = 2   # Number of input features (for XOR, it's 2)
hidden_size = 4  # Number of hidden units
output_size = 2  # Number of classes (binary classification)

# Instantiate the network
model = TwoLayerNet(input_size, hidden_size, output_size)

# Define loss function (cross-entropy for classification)
criterion = nn.CrossEntropyLoss()

# Define optimizer (stochastic gradient descent)
optimizer = optim.SGD(model.parameters(), lr=0.1)

# Train the model
train(model, criterion, optimizer, X_train, y_train, epochs=1000)

# Make predictions
with torch.no_grad():  # No need to track gradients for inference
    outputs = model(X_train)
    _, predicted = torch.max(outputs, 1)
    print('Predicted labels:', predicted.numpy())

Epoch [100/1000], Loss: 0.5649
Epoch [200/1000], Loss: 0.4691
Epoch [300/1000], Loss: 0.2323
Epoch [400/1000], Loss: 0.0977
Epoch [500/1000], Loss: 0.0558
Epoch [600/1000], Loss: 0.0379
Epoch [700/1000], Loss: 0.0282
Epoch [800/1000], Loss: 0.0223
Epoch [900/1000], Loss: 0.0183
Epoch [1000/1000], Loss: 0.0154
Predicted labels: [0 1 1 0]


### Sequence Classification

In [2]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset, DataLoader

# Define the dataset
class CustomDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Initialize tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Prepare data
train_texts = ["Example sentence 1", "Example sentence 2"]
train_labels = [0, 1]
val_texts = ["Example sentence 3", "Example sentence 4"]
val_labels = [0, 1]

train_dataset = CustomDataset(train_texts, train_labels, tokenizer, max_length=128)
val_dataset = CustomDataset(val_texts, val_labels, tokenizer, max_length=128)

# Set training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    evaluation_strategy="epoch"
)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

# Train the model
trainer.train()

# Evaluate the model
eval_result = trainer.evaluate()
print(f"Validation loss: {eval_result['eval_loss']}")

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForSequenceClassification: ['cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.weight', 'cls.predictions.bias', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly i

Epoch,Training Loss,Validation Loss
1,No log,0.767513
2,No log,0.767249
3,No log,0.766758


Validation loss: 0.766758382320404


In [3]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification

def predict_sequence_class(text):
    # Tokenize and prepare input
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    
    # Perform inference
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Get predicted label
    predicted_class = torch.argmax(outputs.logits, dim=1).item()
    return predicted_class

# Example usage
text = "This is an example sentence."
predicted_class = predict_sequence_class(text)
print(f"Predicted class: {predicted_class}")

Predicted class: 1


### Token Classification

In [4]:
import torch
from transformers import BertTokenizer, BertForTokenClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset, DataLoader

# Define the dataset
class CustomTokenDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            is_split_into_words=True,
            return_tensors="pt"
        )
        labels = [-100] * self.max_length
        label_ids = label[:self.max_length] + [-100] * (self.max_length - len(label))
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(label_ids, dtype=torch.long)
        }

# Initialize tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForTokenClassification.from_pretrained('bert-base-uncased', num_labels=3)

# Prepare data
train_texts = [["Hello", "world"], ["This", "is", "a", "test"]]
train_labels = [[1, 0], [2, 0, 0, 1]]
val_texts = [["Another", "example"], ["Token", "classification"]]
val_labels = [[1, 0], [2, 1]]

train_dataset = CustomTokenDataset(train_texts, train_labels, tokenizer, max_length=10)
val_dataset = CustomTokenDataset(val_texts, val_labels, tokenizer, max_length=10)

# Set training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    evaluation_strategy="epoch"
)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

# Train the model
trainer.train()

# Evaluate the model
eval_result = trainer.evaluate()
print(f"Validation loss: {eval_result['eval_loss']}")

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForTokenClassification: ['cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.weight', 'cls.predictions.bias', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: 

Epoch,Training Loss,Validation Loss
1,No log,1.177087
2,No log,1.176900
3,No log,1.176560


Validation loss: 1.1765598058700562


In [5]:
import torch
from transformers import BertTokenizer, BertForTokenClassification

def predict_token_labels(tokens):
    # Tokenize and prepare input
    inputs = tokenizer(tokens, is_split_into_words=True, return_tensors="pt", truncation=True, padding=True, max_length=128)
    
    # Perform inference
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Get predicted labels for each token
    predictions = torch.argmax(outputs.logits, dim=2)
    
    # Map predictions back to tokens
    token_labels = [pred.item() for pred in predictions[0]]
    
    # Filter out padding tokens
    token_labels = token_labels[:len(tokens)]
    return token_labels

# Example usage
tokens = ["This", "is", "an", "example", "sentence", "."]
predicted_labels = predict_token_labels(tokens)
print(f"Tokens: {tokens}")
print(f"Predicted labels: {predicted_labels}")

Tokens: ['This', 'is', 'an', 'example', 'sentence', '.']
Predicted labels: [2, 2, 2, 0, 0, 2]
